# 01 Data pull, freeze and quality checks

Work packages 2.2 and 2.3 of the WBS. This notebook:

1. downloads daily prices for the 16 ETFs from Yahoo Finance (2009-01-02 to 2026-09-30),
2. freezes the download on Google Drive with a checksum,
3. runs the quality checks from section 2.9 of the data specification,
4. saves a clean, aligned daily table that every later notebook starts from.

Run the cells from top to bottom. The download runs only once; later runs load the frozen file.

In [ ]:
# Setup: run this cell first in every Colab session.
# It mounts Google Drive (where the data lives), gets the latest code from
# GitHub (where the code lives) and installs the pinned libraries.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# First run in a session: clone the repo. Later runs: pull the newest code.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make "from src import ..." work.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
# Settings and folders. All values come from src/config.py.
import importlib

import pandas as pd
import yfinance as yf

from src import config, data_pull, quality

# Reload the modules so edits pulled from GitHub take effect without
# restarting the runtime. Order matters: config first, quality last.
for module in (config, data_pull, quality):
    importlib.reload(module)

DIRS = config.data_dirs()

print("Tickers:", len(config.ALL_TICKERS), config.ALL_TICKERS)
print("Window: ", config.START, "to", config.END, "(end exclusive)")
print("yfinance", yf.__version__, "(expected", config.YFINANCE_VERSION + ")")
for name, path in DIRS.items():
    print(f"{name:>9}: {path}")

## Step 1. Download and freeze

Runs only when no freeze exists yet. To make a new freeze on purpose (for example after changing tickers), set `DOWNLOAD_NEW = True`, run the cell, then set it back to `False`.

In [ ]:
DOWNLOAD_NEW = False

existing = data_pull.list_freezes(DIRS["raw"])
if existing and not DOWNLOAD_NEW:
    print("Freeze already exists, skipping the download:", existing[-1].name)
else:
    prices_raw = data_pull.download_prices()
    data_pull.freeze(prices_raw, DIRS["raw"])

## Step 2. Load the frozen file

From here on, everything uses the frozen file. The checksum check stops the notebook if the file was changed after freezing.

In [ ]:
prices, meta = data_pull.load_frozen(DIRS["raw"])

print("Downloaded:", meta["downloaded_utc"], "with yfinance", meta["versions"]["yfinance"])
prices.head()

## Step 3. Quality checks

PASS needs nothing. WARN is either fixed by the cleaning step or worth one look (read every WARN line once). FAIL stops the notebook: fix it before going on, usually by re-downloading.

In [ ]:
report = quality.run_quality_checks(prices)
overall = quality.summarize(report)

stamp = meta["file"].replace("prices_raw_", "").replace(".csv", "")
report.to_csv(DIRS["reports"] / f"quality_report_{stamp}.csv", index=False)

if overall == "FAIL":
    raise RuntimeError("At least one check failed. Read the FAIL lines above before cleaning.")

## Step 4. Clean and save

Aligns all tickers on one trading calendar, fills single missing days from the previous day (logged), flags zero-volume days, and checks every month-end has a price.

In [ ]:
clean, fill_log = quality.clean_prices(prices)

month_end_report = quality.check_month_ends(clean)
quality.summarize(month_end_report)
if (month_end_report["status"] == "FAIL").any():
    raise RuntimeError("Missing month-end prices. Do not build features on this table.")

quality.save_clean(clean, fill_log, DIRS["processed"], DIRS["reports"])
print(f"{len(fill_log)} filled prices (see fill_log_v1.csv)")

In [ ]:
# Coverage table: one row per ticker. Paste it into the data collection sheet.
coverage = quality.coverage_table(clean)
coverage.to_csv(DIRS["reports"] / "coverage_v1.csv", index=False)
coverage

## Step 5. Visual check

Growth of 1 dollar for each ETF on a log scale. Look for jumps that are not real market events (a bad adjustment shows up as a vertical step). Expected: the 2020 crash in every equity line, AGG flat, DBC volatile.

In [ ]:
import matplotlib.pyplot as plt

adj = clean.pivot(index="date", columns="ticker", values="adj_close")
growth = adj / adj.iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (name, tickers) in zip(axes, [("ETF_A", config.ETF_A), ("ETF_B", config.ETF_B)]):
    growth[tickers].plot(ax=ax, logy=True, linewidth=1)
    ax.set_title(f"{name}: growth of $1 since 2009-01-02 (adjusted)")
    ax.set_xlabel("")
    ax.legend(ncol=2, fontsize=8)
axes[0].set_ylabel("value of $1 (log scale)")
plt.tight_layout()
plt.show()

## Step 6. Optional cross-check with a second source

Download a daily CSV for SPY, AGG and JPXN from another source (for example stooq.com/q/d/?s=spy.us), upload them to `data/raw/` on Drive as `stooq_spy.csv` and so on, and run this cell. Missing files are skipped.

In [ ]:
rows = []
for ticker in ["SPY", "AGG", "JPXN"]:
    path = DIRS["raw"] / f"stooq_{ticker.lower()}.csv"
    if path.exists():
        rows.append(quality.cross_check(clean, path, ticker))
    else:
        print(f"Skipped {ticker}: {path.name} not found")
if rows:
    display(pd.DataFrame(rows))

## Step 7. Record the freeze

Copy this into the process documentation log and the data collection sheet. Then save the notebook to GitHub: File, Save a copy in GitHub, path `01_data_pull_and_freeze.ipynb`.

In [ ]:
print(f"Freeze file:   {meta['file']}")
print(f"MD5:           {meta['md5']}")
print(f"Downloaded:    {meta['downloaded_utc']}")
print(f"Window:        {meta['start']} to {meta['end_exclusive']} (exclusive)")
print(f"Rows (raw):    {meta['n_rows']:,}")
print(f"Rows (clean):  {len(clean):,}")
print(f"Trading days:  {clean['date'].nunique():,}")
print(f"Month-ends:    {len(quality.month_end_dates(clean))}")
print(f"Checks:        {overall}, filled prices: {len(fill_log)}")